# 14. Event Streaming — Deep Agents 실행을 관찰 가능한 이벤트로 보기

## 학습 목표

- 일반 streaming과 event streaming의 차이를 구분합니다.
- todo, tool, subagent, filesystem 이벤트를 UI/로그 친화적 형태로 정리합니다.
- 실제 장기 실행 agent 없이도 event consumer를 먼저 설계합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

## 14.1 이벤트 모델

Deep Agents frontend와 운영 로그에서는 “최종 답변”보다 중간 이벤트가 중요합니다.

In [ ]:
events = [
    {"type": "todo", "payload": {"task": "outline", "status": "done"}},
    {"type": "tool", "payload": {"name": "read_file", "status": "done"}},
    {"type": "subagent", "payload": {"name": "researcher", "status": "running"}},
    {"type": "message", "payload": {"text": "초안을 작성합니다."}},
]

len(events)

## 14.2 이벤트 consumer 만들기

consumer는 provider별 raw event를 받아 UI나 로그에 필요한 projection으로 바꿉니다.

In [ ]:
def project_event(event: dict) -> str:
    kind = event["type"]
    payload = event["payload"]
    if kind == "tool":
        return f"tool:{payload['name']}:{payload['status']}"
    if kind == "subagent":
        return f"subagent:{payload['name']}:{payload['status']}"
    return f"{kind}:{payload}"

[project_event(event) for event in events]

## 14.3 UI 상태로 누적하기

event stream은 append-only log로 보존하고, 화면 상태는 projection으로 만듭니다.

In [ ]:
state = {"todos": [], "tools": [], "subagents": [], "messages": []}
for event in events:
    bucket = event["type"] + "s"
    if bucket in state:
        state[bucket].append(event["payload"])

state

## 14.4 운영 체크리스트

| 항목 | 질문 |
|---|---|
| event schema | type과 payload가 안정적인가? |
| replay | 같은 event log로 UI 상태를 복원할 수 있는가? |
| privacy | tool input/output에 민감정보가 섞이지 않는가? |
| fallback | streaming 미지원 환경에서 final output만으로 동작하는가? |

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | event log, projection, UI state accumulation |
| **핵심 개념** | Event streaming은 답변 생성보다 “진행 상태를 안전하게 관찰하는 계약”입니다. |
| **다음 단계** | `08_integration/26_deepagents_frontend/` 후보 |

**참고 문서:**
- `docs/deepagents/event-streaming.md`
- `docs/deepagents/streaming.md`
- `docs/deepagents/frontend/subagent-streaming.md`
- `docs/langchain/event-streaming.md`